# 🌊 AI Flood Inundation Mapping — GEE Data Export (Batch: 13 Events with Copernicus EMS Ground Truth)
**UCI CHRS EarthAI Flood Lab | High School Summer Program**

This notebook exports all data needed by the Streamlit dashboard from Google Earth Engine.  
It processes **13 flood events with Copernicus EMS ground truth** automatically — no manual `EVENT_KEY` changes needed.

### Supported Events (13 total)
| Key | Event | Year | EMSR Code |
|-----|-------|------|-----------|
| `harvey` | Hurricane Harvey — Houston, TX | 2017 | EMSR229 |
| `pakistan` | Pakistan Mega Flood — Sindh Province | 2022 | EMSR629 |
| `myanmar2015` | Myanmar Cyclone Komen — Irrawaddy Delta | 2015 | EMSR130 |
| `louisiana2016` | Louisiana Flood — Baton Rouge, LA | 2016 | EMSR176 |
| `srilanka2017` | Sri Lanka Flood — Southern Sri Lanka | 2017 | EMSR205 |
| `mozambique2019` | Cyclone Idai — Beira, Mozambique | 2019 | EMSR348 |
| `iran2019` | Iran Flood — Khuzestan | 2019 | EMSR352 |
| `germany2021` | Ahr Valley Flood — Rhineland-Palatinate | 2021 | EMSR517 |
| `nigeria2022` | Nigeria Flood — Anambra/Delta State | 2022 | EMSR314 |
| `libya2023` | Libya Flood (Derna) | 2023 | EMSR696 |
| `somalia2023` | Somalia Flood — Hirshabelle | 2023 | EMSR404 |
| `brazil2024` | Brazil Rio Grande Flood | 2024 | EMSR720 |
| `valencia2024` | Spain Valencia Flood | 2024 | EMSR773 |

### How to Run
1. **Cell 1** — Install packages  
2. **Cell 2** — Authenticate with GEE  
3. **Cell 3** — Review/override global config  
4. **Cell 3b** — Prepare Copernicus EMS ground truth assets (one-time teacher setup)  
5. **Cell 4** — Define all processing functions (no output)  
6. **Cell 5** — ▶ Run batch loop — processes & exports all 13 events  
7. **Cell 6** — Monitor all export tasks  

> 💡 To process a **subset**, edit `EVENTS_TO_RUN` at the top of Cell 5.

In [3]:
# ── Cell 1: Install Packages ───────────────────────────────────────
# Takes 1–2 minutes on first run. No runtime restart needed.
!pip install -q earthengine-api geemap geopandas rasterio matplotlib tqdm


In [4]:
# ── Cell 2: GEE Authentication ────────────────────────────────────
import ee
import geemap

try:
    ee.Initialize(project='chrs-earthai')  # <-- replace with your GEE project ID
    print('GEE initialized successfully.')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='chrs-earthai')  # <-- same replacement here
    print('GEE authenticated and initialized.')


GEE initialized successfully.


In [ ]:
# ── Cell 3: Global Configuration — 13 Events with Copernicus EMS Ground Truth ──
#
# Parameters per event:
#   emsr          : Copernicus EMS activation code (for ground truth labels)
#   aoi           : bounding box [W, S, E, N]
#   before_start/end : pre-flood SAR/S2 window
#   flood_start/end  : flood SAR/S2 window
#   gpm_start/end    : GPM rainfall CSV window (wider than flood window)
#   center        : map preview center [lon, lat]
#   zoom          : map preview zoom level
#   sar_pass      : 'ASCENDING' | 'DESCENDING' (orbit direction filter)
#   s2_cloud_pct  : Sentinel-2 cloud cover threshold (%)

ALL_EVENTS = {
    'harvey': {
        'name':          'Hurricane Harvey 2017 — Houston, TX',
        'emsr':          'EMSR229',
        'aoi':           [-96.0, 29.0, -94.5, 30.5],
        'before_start':  '2017-08-01', 'before_end':  '2017-08-22',
        'flood_start':   '2017-08-26', 'flood_end':   '2017-09-10',
        'gpm_start':     '2017-08-18', 'gpm_end':     '2017-09-10',
        'center':        [-95.37, 29.76], 'zoom': 9,
        'sar_pass': 'DESCENDING', 's2_cloud_pct': 30,
    },
    'pakistan': {
        'name':          'Pakistan Mega Flood 2022 — Sindh Province',
        'emsr':          'EMSR629',
        'aoi':           [66.5, 25.5, 69.5, 28.0],
        'before_start':  '2022-06-01', 'before_end':  '2022-07-20',
        'flood_start':   '2022-08-15', 'flood_end':   '2022-09-25',
        'gpm_start':     '2022-07-20', 'gpm_end':     '2022-09-25',
        'center':        [68.0, 27.0], 'zoom': 8,
        'sar_pass': 'ASCENDING', 's2_cloud_pct': 15,
    },
    'myanmar2015': {
        'name':          'Myanmar Cyclone Komen 2015 — Irrawaddy Delta',
        'emsr':          'EMSR130',
        'aoi':           [94.5, 15.5, 96.5, 17.5],
        'before_start':  '2015-06-01', 'before_end':  '2015-07-20',
        'flood_start':   '2015-07-25', 'flood_end':   '2015-08-20',
        'gpm_start':     '2015-07-10', 'gpm_end':     '2015-08-20',
        'center':        [95.5, 16.5], 'zoom': 8,
        'sar_pass': 'ASCENDING', 's2_cloud_pct': 30,
    },
    'louisiana2016': {
        'name':          'Louisiana Flood 2016 — Baton Rouge, LA',
        'emsr':          'EMSR176',
        'aoi':           [-91.5, 30.0, -90.0, 31.0],
        'before_start':  '2016-07-01', 'before_end':  '2016-08-10',
        'flood_start':   '2016-08-12', 'flood_end':   '2016-08-25',
        'gpm_start':     '2016-08-05', 'gpm_end':     '2016-08-25',
        'center':        [-90.75, 30.5], 'zoom': 9,
        'sar_pass': 'DESCENDING', 's2_cloud_pct': 30,
    },
    'srilanka2017': {
        'name':          'Sri Lanka Flood 2017 — Southern Sri Lanka',
        'emsr':          'EMSR205',
        'aoi':           [80.0, 6.0, 81.5, 7.5],
        'before_start':  '2017-04-01', 'before_end':  '2017-05-15',
        'flood_start':   '2017-05-25', 'flood_end':   '2017-06-15',
        'gpm_start':     '2017-05-15', 'gpm_end':     '2017-06-15',
        'center':        [80.75, 6.75], 'zoom': 9,
        'sar_pass': 'ASCENDING', 's2_cloud_pct': 25,
    },
    'mozambique2019': {
        'name':          'Cyclone Idai 2019 — Beira, Mozambique',
        'emsr':          'EMSR348',
        'aoi':           [34.0, -20.0, 35.5, -18.5],
        'before_start':  '2019-02-01', 'before_end':  '2019-03-10',
        'flood_start':   '2019-03-14', 'flood_end':   '2019-04-01',
        'gpm_start':     '2019-03-05', 'gpm_end':     '2019-04-01',
        'center':        [34.75, -19.25], 'zoom': 9,
        'sar_pass': 'DESCENDING', 's2_cloud_pct': 25,
    },
    'iran2019': {
        'name':          'Iran Flood 2019 — Khuzestan',
        'emsr':          'EMSR352',
        'aoi':           [48.0, 31.0, 50.0, 33.0],
        'before_start':  '2019-02-01', 'before_end':  '2019-03-20',
        'flood_start':   '2019-03-25', 'flood_end':   '2019-04-20',
        'gpm_start':     '2019-03-15', 'gpm_end':     '2019-04-20',
        'center':        [49.0, 32.0], 'zoom': 8,
        'sar_pass': 'ASCENDING', 's2_cloud_pct': 15,
    },
    'germany2021': {
        'name':          'Ahr Valley Flood 2021 — Rhineland-Palatinate',
        'emsr':          'EMSR517',
        'aoi':           [6.5, 50.0, 7.5, 50.8],
        'before_start':  '2021-06-01', 'before_end':  '2021-07-12',
        'flood_start':   '2021-07-14', 'flood_end':   '2021-07-25',
        'gpm_start':     '2021-07-10', 'gpm_end':     '2021-07-25',
        'center':        [7.0, 50.4], 'zoom': 10,
        'sar_pass': 'DESCENDING', 's2_cloud_pct': 25,
    },
    'nigeria2022': {
        'name':          'Nigeria Flood 2022 — Anambra/Delta State',
        'emsr':          'EMSR314',
        'aoi':           [6.0, 5.0, 7.5, 6.5],
        'before_start':  '2022-08-01', 'before_end':  '2022-09-15',
        'flood_start':   '2022-09-20', 'flood_end':   '2022-10-20',
        'gpm_start':     '2022-09-10', 'gpm_end':     '2022-10-20',
        'center':        [6.75, 5.75], 'zoom': 9,
        'sar_pass': 'DESCENDING', 's2_cloud_pct': 25,
    },
    'libya2023': {
        'name':          'Libya Flood 2023 — Derna',
        'emsr':          'EMSR696',
        'aoi':           [22.0, 32.0, 23.5, 33.0],
        'before_start':  '2023-08-01', 'before_end':  '2023-09-09',
        'flood_start':   '2023-09-11', 'flood_end':   '2023-09-25',
        'gpm_start':     '2023-09-05', 'gpm_end':     '2023-09-25',
        'center':        [22.75, 32.5], 'zoom': 10,
        'sar_pass': 'ASCENDING', 's2_cloud_pct': 10,
    },
    'somalia2023': {
        'name':          'Somalia Flood 2023 — Hirshabelle',
        'emsr':          'EMSR404',
        'aoi':           [45.0, 2.0, 46.5, 3.5],
        'before_start':  '2023-10-01', 'before_end':  '2023-11-01',
        'flood_start':   '2023-11-05', 'flood_end':   '2023-11-30',
        'gpm_start':     '2023-10-25', 'gpm_end':     '2023-11-30',
        'center':        [45.75, 2.75], 'zoom': 9,
        'sar_pass': 'ASCENDING', 's2_cloud_pct': 20,
    },
    'brazil2024': {
        'name':          'Brazil Rio Grande Flood 2024',
        'emsr':          'EMSR720',
        'aoi':           [-53.0, -31.0, -51.0, -29.5],
        'before_start':  '2024-04-01', 'before_end':  '2024-04-30',
        'flood_start':   '2024-05-01', 'flood_end':   '2024-05-25',
        'gpm_start':     '2024-04-25', 'gpm_end':     '2024-05-25',
        'center':        [-52.0, -30.25], 'zoom': 8,
        'sar_pass': 'ASCENDING', 's2_cloud_pct': 25,
    },
    'valencia2024': {
        'name':          'Spain Valencia Flood 2024',
        'emsr':          'EMSR773',
        'aoi':           [-1.5, 38.5, 0.5, 40.0],
        'before_start':  '2024-10-01', 'before_end':  '2024-10-28',
        'flood_start':   '2024-10-29', 'flood_end':   '2024-11-10',
        'gpm_start':     '2024-10-25', 'gpm_end':     '2024-11-10',
        'center':        [-0.5, 39.25], 'zoom': 9,
        'sar_pass': 'DESCENDING', 's2_cloud_pct': 20,
    },
}

# Export resolution settings
SAR_SCALE = 20   # meters (SAR native 10m, downsampled)
OPT_SCALE = 30   # meters (S2 native 10m)
DEM_SCALE = 30   # meters (SRTM native 30m)

print(f'Config loaded: {len(ALL_EVENTS)} events defined.')
print('Events:', ', '.join(ALL_EVENTS.keys()))

### Cell 3b: Preparing Copernicus EMS Ground Truth

The RF training labels use **Copernicus EMS flood delineation polygons** as external ground truth.

**One-time setup (teacher):**
1. Visit the Copernicus EMS activation page for each event (e.g., `mapping.emergency.copernicus.eu/activations/EMSR229/`)
2. Download the **Delineation Vector Package** (ZIP containing shapefiles)
3. Extract the `*_observed_event_*.shp` file (the flood extent polygon)
4. Upload to GEE as an asset: `projects/chrs-earthai/assets/ems/{EMSR_CODE}_flood`
   - Use the GEE Code Editor: Assets tab → NEW → Shape files → Upload

| Event | EMSR | Upload Asset Path |
|-------|------|-------------------|
| harvey | EMSR229 | `projects/chrs-earthai/assets/ems/EMSR229_flood` |
| pakistan | EMSR629 | `projects/chrs-earthai/assets/ems/EMSR629_flood` |
| myanmar2015 | EMSR130 | `projects/chrs-earthai/assets/ems/EMSR130_flood` |
| louisiana2016 | EMSR176 | `projects/chrs-earthai/assets/ems/EMSR176_flood` |
| srilanka2017 | EMSR205 | `projects/chrs-earthai/assets/ems/EMSR205_flood` |
| mozambique2019 | EMSR348 | `projects/chrs-earthai/assets/ems/EMSR348_flood` |
| iran2019 | EMSR352 | `projects/chrs-earthai/assets/ems/EMSR352_flood` |
| germany2021 | EMSR517 | `projects/chrs-earthai/assets/ems/EMSR517_flood` |
| nigeria2022 | EMSR314 | `projects/chrs-earthai/assets/ems/EMSR314_flood` |
| libya2023 | EMSR696 | `projects/chrs-earthai/assets/ems/EMSR696_flood` |
| somalia2023 | EMSR404 | `projects/chrs-earthai/assets/ems/EMSR404_flood` |
| brazil2024 | EMSR720 | `projects/chrs-earthai/assets/ems/EMSR720_flood` |
| valencia2024 | EMSR773 | `projects/chrs-earthai/assets/ems/EMSR773_flood` |

> ⚠️ Events without uploaded EMS assets will skip RF sample generation.

In [ ]:
# ── Cell 4: Define All Processing Functions ───────────────────────
# No output is produced here. All functions are used by the batch loop in Cell 5.

import datetime, os, shutil
import pandas as pd
from tqdm import tqdm

# ── Mount Google Drive once ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Final dataset root for dashboard ──────────────────────────────
DRIVE_DATA_ROOT = '/content/drive/MyDrive/CHRS_EarthAI/data'
os.makedirs(DRIVE_DATA_ROOT, exist_ok=True)

def get_aoi(cfg):
    """Convert [W, S, E, N] bbox to ee.Geometry.Rectangle."""
    b = cfg['aoi']
    return ee.Geometry.Rectangle([b[0], b[1], b[2], b[3]])


def load_sar(aoi, cfg, start, end):
    """Load and mosaic Sentinel-1 SAR VH backscatter over the AOI."""
    col = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .filter(ee.Filter.eq('orbitProperties_pass', cfg['sar_pass']))
        .select('VH')
    )
    count = col.size().getInfo()
    if count == 0:
        print(f'    [SAR] No images with {cfg["sar_pass"]} pass — retrying without orbit filter...')
        col = (
            ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .select('VH')
        )
        count = col.size().getInfo()
        print(f'    [SAR] Found {count} images (both orbit directions)')
    else:
        print(f'    [SAR] {count} scenes found ({start} → {end})')
    return col.mean().clip(aoi)


def load_s2(aoi, cfg, start, end):
    """Load Sentinel-2 SR median composite with NDWI/MNDWI bands."""
    col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cfg['s2_cloud_pct']))
        .select(['B3', 'B4', 'B8', 'B11'])
        .median()
        .clip(aoi)
    )
    ndwi  = col.normalizedDifference(['B3', 'B8']).rename('NDWI')
    mndwi = col.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    return col.addBands([ndwi, mndwi])


def load_terrain(aoi):
    """Load SRTM DEM + slope + JRC permanent water."""
    dem             = ee.Image('USGS/SRTMGL1_003').clip(aoi).rename('elevation')
    slope           = ee.Terrain.slope(dem).rename('slope')
    jrc             = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').clip(aoi)
    permanent_water = jrc.select('occurrence').gt(90).rename('permanent_water')
    return dem, slope, permanent_water


def extract_gpm(aoi, cfg, event_key):
    """Extract daily GPM rainfall CSV and save locally to /content/."""
    gpm_col = (
        ee.ImageCollection('NASA/GPM_L3/IMERG_V07')
        .filterBounds(aoi)
        .filterDate(cfg['gpm_start'], cfg['gpm_end'])
        .select('precipitation')
    )
    start_dt = datetime.datetime.strptime(cfg['gpm_start'], '%Y-%m-%d')
    end_dt   = datetime.datetime.strptime(cfg['gpm_end'],   '%Y-%m-%d')
    dates    = pd.date_range(start_dt, end_dt, freq='D')

    records, skipped = [], []
    for dt in dates:
        d_str  = dt.strftime('%Y-%m-%d')
        d_next = (dt + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
        daily_col = gpm_col.filterDate(d_str, d_next)
        if daily_col.size().getInfo() == 0:
            skipped.append(d_str)
            records.append({'date': d_str, 'precip_mm_day': 0.0, 'event': event_key})
            continue
        stats = (
            daily_col.mean()
            .multiply(24)
            .reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi,
                          scale=11132, maxPixels=1e9)
        )
        val = stats.get('precipitation').getInfo()
        records.append({'date': d_str, 'precip_mm_day': val if val is not None else 0.0,
                        'event': event_key})

    gpm_df = pd.DataFrame(records)
    gpm_df['precip_mm_day'] = gpm_df['precip_mm_day'].fillna(0).round(2)
    if skipped:
        print(f'    [GPM] {len(skipped)} days had no granules (set to 0)')
    peak_row = gpm_df.loc[gpm_df['precip_mm_day'].idxmax()]
    print(f'    [GPM] {len(gpm_df)} records | peak {peak_row["precip_mm_day"]:.1f} mm/day on {peak_row["date"]}')

    local_path = f'/content/{event_key}_GPM_rainfall_daily.csv'
    gpm_df.to_csv(local_path, index=False)
    return local_path


# ── EMSR asset path pattern ──────────────────────────────────────
EMS_ASSET_PREFIX = 'projects/chrs-earthai/assets/ems'

def load_ems_flood_mask(event_key, cfg, aoi):
    """Load Copernicus EMS flood delineation polygons from GEE asset.

    Teacher must upload EMS shapefiles to GEE assets beforehand.
    Asset path: projects/chrs-earthai/assets/ems/{EMSR_CODE}_flood
    """
    emsr = cfg['emsr']
    asset_id = f'{EMS_ASSET_PREFIX}/{emsr}_flood'
    try:
        fc = ee.FeatureCollection(asset_id)
        # Rasterize: 1 inside flood polygons, 0 outside
        flood_img = fc.reduceToImage([], ee.Reducer.first()).unmask(0).gt(0).rename('label')
        return flood_img.clip(aoi)
    except Exception as e:
        print(f'    [EMS] WARNING: Asset not found: {asset_id}')
        print(f'    [EMS] Upload EMS shapefile for {emsr} to this asset path.')
        return None


def build_rf_samples(aoi, cfg, event_key, sar_after, s2_after, dem, slope, permanent_water):
    """Build RF training samples using Copernicus EMS ground truth labels.

    Labels come from EMS flood delineation polygons (external ground truth),
    NOT from SAR thresholding. This avoids circular reasoning where the model
    learns to reproduce the same threshold used to generate labels.
    """
    flood_mask = load_ems_flood_mask(event_key, cfg, aoi)
    if flood_mask is None:
        print(f'    [RF] SKIPPED — no EMS ground truth for {event_key}')
        return None

    # Remove permanent water from flood mask
    flood_label = flood_mask.And(permanent_water.Not()).rename('label')

    # Non-flood: everything NOT in flood polygons AND NOT permanent water
    nonflood_label = flood_mask.Not().And(permanent_water.Not())

    # Combined label: 1=flood, 0=non-flood (masked to exclude permanent water)
    label_img = (
        ee.Image(0)
        .where(flood_label, 1)
        .updateMask(flood_label.Or(nonflood_label))
        .rename('label')
        .toUint8()
    )

    feature_stack = ee.Image.cat([
        sar_after.rename('SAR_VH'),
        s2_after.select('NDWI'),
        s2_after.select('MNDWI'),
        dem,
        slope,
        permanent_water
    ])

    fc = (
        feature_stack.addBands(label_img)
        .stratifiedSample(
            numPoints=400, classBand='label', region=aoi,
            scale=30, seed=2024, geometries=False
        )
    )
    n = fc.size().getInfo()
    print(f'    [RF] {n} training samples with EMS ground truth (target ~800)')
    if n < 200:
        print('    [RF] WARNING: low sample count — check EMS asset coverage')
    return fc


def queue_exports(event_key, cfg, aoi,
                  sar_before, sar_after,
                  s2_before, s2_after,
                  dem, slope, permanent_water,
                  training_fc):
    """Start all GEE export tasks for one event. Returns list of (name, task)."""

    # GEE folder cannot be a nested Drive path.
    # Use a unique flat folder name, then move files later.
    export_folder = f'CHRS_EarthAI_export_{event_key}'

    def exp_img(image, name, scale):
        t = ee.batch.Export.image.toDrive(
            image=image.toFloat(),
            description=f'{event_key}_{name}',
            folder=export_folder,
            fileNamePrefix=f'{event_key}_{name}',
            region=aoi,
            scale=scale,
            maxPixels=1e10,
            fileFormat='GeoTIFF'
        )
        t.start()
        return t

    def exp_tbl(fc, name):
        t = ee.batch.Export.table.toDrive(
            collection=fc,
            description=f'{event_key}_{name}',
            folder=export_folder,
            fileNamePrefix=f'{event_key}_{name}',
            fileFormat='CSV'
        )
        t.start()
        return t

    return [
        ('SAR_before',          exp_img(sar_before,                                'SAR_before',          SAR_SCALE)),
        ('SAR_after',           exp_img(sar_after,                                 'SAR_after',           SAR_SCALE)),
        ('NDWI_before',         exp_img(s2_before.select('NDWI'),                  'NDWI_before',         OPT_SCALE)),
        ('NDWI_after',          exp_img(s2_after.select('NDWI'),                   'NDWI_after',          OPT_SCALE)),
        ('RGB_before',          exp_img(s2_before.select(['B4','B3','B8']).toUint16(), 'RGB_before',       OPT_SCALE)),
        ('RGB_after',           exp_img(s2_after.select(['B4','B3','B8']).toUint16(),  'RGB_after',        OPT_SCALE)),
        ('DEM_slope',           exp_img(dem.addBands(slope),                       'DEM_slope',           DEM_SCALE)),
        ('JRC_permanent_water', exp_img(permanent_water.toUint8(),                 'JRC_permanent_water', 30)),
        ('RF_training_samples', exp_tbl(training_fc,                               'RF_training_samples')),
    ]


def copy_gpm_to_drive(event_key, local_csv_path):
    """Copy GPM CSV directly into final dashboard data folder."""
    drive_path = os.path.join(DRIVE_DATA_ROOT, event_key)
    os.makedirs(drive_path, exist_ok=True)

    dst = os.path.join(drive_path, f'{event_key}_GPM_rainfall_daily.csv')
    shutil.copy(local_csv_path, dst)
    print(f'    [GPM] CSV → Drive: {dst}')


print('All processing functions defined. Ready to run batch loop (Cell 5).')

In [ ]:
# ── Cell 5: Batch Loop — Process All 13 Events ────────────────────
#
# This cell iterates over every event in ALL_EVENTS and:
#   1. Computes SAR, S2, terrain, GPM, RF layers
#   2. Queues GEE export tasks (GeoTIFFs + RF CSV) to Google Drive
#   3. Copies GPM rainfall CSV directly to Google Drive
#
# RF labels use Copernicus EMS ground truth (not heuristic pseudo-labels).
# Events without uploaded EMS assets will skip RF sample generation.
#
# GEE exports run asynchronously — monitor progress in Cell 6.
#
# ── To process a subset, edit EVENTS_TO_RUN ──────────────────────
EVENTS_TO_RUN = list(ALL_EVENTS.keys())   # all 13 events
# EVENTS_TO_RUN = ['harvey', 'pakistan']  # example: subset

# ── Skip already-exported events (check Drive folder existence) ───
SKIP_EXISTING = True   # set False to re-export everything

all_tasks  = {}   # event_key → [(name, task), ...]
failed_events = []

total = len(EVENTS_TO_RUN)
for idx, event_key in enumerate(EVENTS_TO_RUN):
    cfg = ALL_EVENTS[event_key]
    print(f'\n[{idx+1}/{total}] {event_key} — {cfg["name"]}')
    print('─' * 60)

    # ── Skip check ────────────────────────────────────────────────
    if SKIP_EXISTING:
        drive_dir = os.path.join(DRIVE_DATA_ROOT, event_key)
        if os.path.isdir(drive_dir):
            existing = os.listdir(drive_dir)
            if len(existing) >= 9:   # 8 tifs + RF csv minimum
                print(f'  SKIP — Drive folder already has {len(existing)} files.')
                continue
            print(f'  Drive folder exists but only {len(existing)} files — re-exporting.')

    try:
        aoi = get_aoi(cfg)

        # ── SAR ───────────────────────────────────────────────────
        print('  Processing SAR...')
        sar_before = load_sar(aoi, cfg, cfg['before_start'], cfg['before_end'])
        sar_after  = load_sar(aoi, cfg, cfg['flood_start'],  cfg['flood_end'])
        sar_diff   = sar_after.subtract(sar_before).rename('SAR_diff')

        # ── Sentinel-2 ────────────────────────────────────────────
        print('  Processing Sentinel-2...')
        s2_before  = load_s2(aoi, cfg, cfg['before_start'], cfg['before_end'])
        s2_after   = load_s2(aoi, cfg, cfg['flood_start'],  cfg['flood_end'])

        # ── Terrain + JRC ─────────────────────────────────────────
        print('  Processing terrain...')
        dem, slope, permanent_water = load_terrain(aoi)

        # ── GPM rainfall CSV ──────────────────────────────────────
        print('  Extracting GPM rainfall...')
        local_gpm = extract_gpm(aoi, cfg, event_key)

        # ── RF training samples (EMS ground truth) ────────────────
        print('  Building RF training samples...')
        training_fc = build_rf_samples(
            aoi, cfg, event_key, sar_after,
            s2_after, dem, slope, permanent_water
        )

        # ── Queue GEE exports ─────────────────────────────────────
        if training_fc is not None:
            print('  Queuing GEE export tasks...')
            tasks = queue_exports(
                event_key, cfg, aoi,
                sar_before, sar_after,
                s2_before, s2_after,
                dem, slope, permanent_water,
                training_fc
            )
            all_tasks[event_key] = tasks
            for name, _ in tasks:
                print(f'    Queued: {event_key}_{name}')
        else:
            print(f'  ⚠  Skipping RF export — no EMS ground truth')
            # Still export rasters (SAR, S2, DEM, JRC) without RF samples
            print('  Queuing GEE export tasks (without RF samples)...')
            tasks = queue_exports(
                event_key, cfg, aoi,
                sar_before, sar_after,
                s2_before, s2_after,
                dem, slope, permanent_water,
                ee.FeatureCollection([])  # empty placeholder
            )
            # Remove the RF export task (last one) since we have no labels
            tasks = tasks[:-1]
            all_tasks[event_key] = tasks
            for name, _ in tasks:
                print(f'    Queued: {event_key}_{name}')

        # ── Copy GPM CSV to Drive (immediate) ─────────────────────
        copy_gpm_to_drive(event_key, local_gpm)

        print(f'  ✓  {event_key} done — {len(tasks)} export tasks queued.')

    except Exception as e:
        print(f'  ✗  {event_key} FAILED: {e}')
        failed_events.append(event_key)
        import traceback; traceback.print_exc()

# ── Summary ───────────────────────────────────────────────────────
print('\n' + '═' * 60)
total_tasks = sum(len(v) for v in all_tasks.values())
print(f'Batch complete.')
print(f'  Events queued : {len(all_tasks)}')
print(f'  Total tasks   : {total_tasks}')
if failed_events:
    print(f'  FAILED events : {failed_events}')
print('Run Cell 6 to monitor GEE export progress.')

In [ ]:
# ── Cell 6: Monitor All Export Tasks ─────────────────────────────
# Polls every 30 seconds until all tasks finish.
# Alternatively, check: https://code.earthengine.google.com/tasks

import time

def monitor_all_tasks(all_tasks_dict, interval_sec=30):
    """Monitor all queued GEE tasks across all events."""
    flat_tasks  = []
    flat_names  = []
    for ev, task_list in all_tasks_dict.items():
        for name, t in task_list:
            flat_tasks.append(t)
            flat_names.append(f'{ev}/{name}')

    terminal = {'COMPLETED', 'FAILED', 'CANCELLED'}
    total    = len(flat_tasks)

    print(f'Monitoring {total} tasks across {len(all_tasks_dict)} events...')
    print('─' * 60)

    while True:
        statuses = [t.status()['state'] for t in flat_tasks]
        done     = sum(1 for s in statuses if s in terminal)
        failed   = [flat_names[i] for i, s in enumerate(statuses) if s == 'FAILED']
        running  = sum(1 for s in statuses if s == 'RUNNING')
        pending  = sum(1 for s in statuses if s in ('READY', 'UNSUBMITTED'))

        line = (f'[{time.strftime("%H:%M:%S")}]  '
                f'{done}/{total} done  |  {running} running  |  {pending} queued')
        if failed:
            line += f'  |  FAILED: {len(failed)}'
        print(line)

        if done == total:
            break
        time.sleep(interval_sec)

    print('\nAll tasks finished.')
    print('─' * 60)
    ok_count = fail_count = 0
    for name, status in zip(flat_names, statuses):
        icon = '✓' if status == 'COMPLETED' else '✗'
        if status == 'COMPLETED':
            ok_count += 1
        else:
            fail_count += 1
            print(f'  [{icon}] {name}: {status}')
    print(f'\nResult: {ok_count} completed, {fail_count} failed.')

if all_tasks:
    monitor_all_tasks(all_tasks)
else:
    print('No tasks in all_tasks dict. Run Cell 5 first.')


Monitoring 144 tasks across 16 events...
────────────────────────────────────────────────────────────


[04:30:54]  4/144 done  |  1 running  |  139 queued
[04:32:05]  5/144 done  |  1 running  |  138 queued
[04:33:11]  5/144 done  |  1 running  |  138 queued
[04:34:15]  5/144 done  |  1 running  |  138 queued
[04:35:22]  5/144 done  |  1 running  |  138 queued
[04:36:28]  5/144 done  |  1 running  |  138 queued
[04:37:34]  5/144 done  |  1 running  |  138 queued
[04:38:40]  5/144 done  |  1 running  |  138 queued
[04:39:44]  6/144 done  |  1 running  |  137 queued
[04:40:49]  6/144 done  |  1 running  |  137 queued
[04:41:54]  6/144 done  |  1 running  |  137 queued
[04:43:01]  6/144 done  |  1 running  |  137 queued
[04:44:05]  6/144 done  |  1 running  |  137 queued
[04:45:08]  7/144 done  |  1 running  |  136 queued
[04:46:14]  7/144 done  |  1 running  |  136 queued
[04:47:18]  7/144 done  |  1 running  |  136 queued
[04:48:23]  7/144 done  |  1 running  |  136 queued
[04:49:30]  9/144 done  |  1 running  |  134 queued
[04:50:35]  9/144 done  |  1 running  |  134 queued
[04:51:39]  

[05:26:42]  11/144 done  |  1 running  |  132 queued
[05:27:47]  11/144 done  |  1 running  |  132 queued
[05:28:51]  11/144 done  |  1 running  |  132 queued
[05:29:56]  11/144 done  |  1 running  |  132 queued
[05:31:12]  11/144 done  |  1 running  |  132 queued
[05:32:19]  11/144 done  |  1 running  |  132 queued
[05:33:25]  11/144 done  |  1 running  |  132 queued
[05:34:30]  11/144 done  |  1 running  |  132 queued
[05:35:34]  11/144 done  |  1 running  |  132 queued
[05:36:40]  12/144 done  |  1 running  |  131 queued
[05:37:45]  12/144 done  |  1 running  |  131 queued
[05:38:50]  12/144 done  |  1 running  |  131 queued
[05:39:55]  12/144 done  |  1 running  |  131 queued
[05:41:00]  12/144 done  |  1 running  |  131 queued
[05:42:21]  12/144 done  |  1 running  |  131 queued
[05:43:27]  12/144 done  |  1 running  |  131 queued
[05:44:32]  12/144 done  |  1 running  |  131 queued
[05:45:37]  13/144 done  |  0 running  |  131 queued
[05:46:42]  13/144 done  |  1 running  |  130 

[06:22:59]  15/144 done  |  1 running  |  128 queued
[06:24:04]  15/144 done  |  1 running  |  128 queued
[06:25:11]  15/144 done  |  1 running  |  128 queued
[06:26:16]  15/144 done  |  1 running  |  128 queued
[06:27:23]  15/144 done  |  1 running  |  128 queued
[06:28:28]  15/144 done  |  1 running  |  128 queued
[06:29:34]  15/144 done  |  1 running  |  128 queued
[06:30:42]  15/144 done  |  1 running  |  128 queued
[06:31:54]  15/144 done  |  1 running  |  128 queued
[06:33:02]  16/144 done  |  1 running  |  127 queued
[06:34:07]  16/144 done  |  1 running  |  127 queued
[06:35:12]  16/144 done  |  1 running  |  127 queued
[06:36:16]  16/144 done  |  1 running  |  127 queued
[06:37:20]  16/144 done  |  1 running  |  127 queued
[06:38:25]  16/144 done  |  1 running  |  127 queued
[06:39:30]  17/144 done  |  1 running  |  126 queued
[06:40:36]  18/144 done  |  1 running  |  125 queued
[06:41:42]  18/144 done  |  1 running  |  125 queued
[06:42:48]  18/144 done  |  1 running  |  125 

[07:19:26]  21/144 done  |  1 running  |  122 queued
[07:20:32]  22/144 done  |  1 running  |  121 queued
[07:21:38]  22/144 done  |  1 running  |  121 queued
[07:22:46]  22/144 done  |  1 running  |  121 queued
[07:23:54]  22/144 done  |  1 running  |  121 queued
[07:24:59]  22/144 done  |  1 running  |  121 queued
[07:26:16]  22/144 done  |  1 running  |  121 queued
[07:27:22]  22/144 done  |  1 running  |  121 queued
[07:28:29]  22/144 done  |  1 running  |  121 queued
[07:29:36]  22/144 done  |  1 running  |  121 queued
[07:30:45]  22/144 done  |  1 running  |  121 queued
[07:31:51]  23/144 done  |  1 running  |  120 queued
[07:33:00]  23/144 done  |  1 running  |  120 queued
[07:34:08]  23/144 done  |  1 running  |  120 queued
[07:35:14]  23/144 done  |  1 running  |  120 queued
[07:36:20]  23/144 done  |  1 running  |  120 queued
[07:37:28]  23/144 done  |  1 running  |  120 queued
[07:38:36]  23/144 done  |  1 running  |  120 queued
[07:39:41]  23/144 done  |  1 running  |  120 

...

[11:05:13]  43/144 done  |  1 running  |  100 queued
[11:06:21]  43/144 done  |  1 running  |  100 queued
[11:07:27]  43/144 done  |  1 running  |  100 queued
[11:08:31]  43/144 done  |  1 running  |  100 queued
[11:09:36]  45/144 done  |  0 running  |  99 queued
[11:10:41]  45/144 done  |  1 running  |  98 queued
[11:11:46]  45/144 done  |  1 running  |  98 queued
[11:12:50]  45/144 done  |  1 running  |  98 queued
[11:13:54]  45/144 done  |  1 running  |  98 queued
[11:14:58]  45/144 done  |  1 running  |  98 queued
[11:16:04]  45/144 done  |  1 running  |  98 queued
[11:17:08]  45/144 done  |  1 running  |  98 queued
[11:18:16]  45/144 done  |  1 running  |  98 queued
[11:19:20]  45/144 done  |  1 running  |  98 queued
[11:20:26]  45/144 done  |  1 running  |  98 queued
[11:21:32]  45/144 done  |  1 running  |  98 queued
[11:22:37]  45/144 done  |  1 running  |  98 queued
[11:23:41]  45/144 done  |  1 running  |  98 queued
[11:24:46]  45/144 done  |  1 running  |  98 queued
[11:25:5

[12:01:48]  48/144 done  |  1 running  |  95 queued
[12:02:52]  48/144 done  |  1 running  |  95 queued
[12:03:56]  48/144 done  |  1 running  |  95 queued
[12:04:59]  48/144 done  |  1 running  |  95 queued
[12:06:05]  48/144 done  |  1 running  |  95 queued
[12:07:08]  48/144 done  |  1 running  |  95 queued
[12:08:13]  48/144 done  |  1 running  |  95 queued
[12:09:17]  48/144 done  |  1 running  |  95 queued
[12:10:23]  48/144 done  |  1 running  |  95 queued
[12:11:28]  48/144 done  |  1 running  |  95 queued
[12:12:32]  48/144 done  |  1 running  |  95 queued
[12:13:38]  49/144 done  |  1 running  |  94 queued
[12:14:42]  49/144 done  |  1 running  |  94 queued
[12:15:49]  49/144 done  |  1 running  |  94 queued
[12:16:53]  49/144 done  |  1 running  |  94 queued
[12:17:59]  49/144 done  |  1 running  |  94 queued
[12:19:04]  49/144 done  |  1 running  |  94 queued
[12:20:12]  49/144 done  |  1 running  |  94 queued
[12:21:18]  49/144 done  |  1 running  |  94 queued
[12:22:23]  

[12:57:36]  51/144 done  |  1 running  |  92 queued
[12:58:42]  51/144 done  |  1 running  |  92 queued
[12:59:48]  51/144 done  |  1 running  |  92 queued
[13:00:54]  51/144 done  |  1 running  |  92 queued
[13:02:01]  51/144 done  |  1 running  |  92 queued
[13:03:08]  51/144 done  |  1 running  |  92 queued
[13:04:12]  51/144 done  |  1 running  |  92 queued
[13:05:19]  51/144 done  |  1 running  |  92 queued
[13:06:24]  51/144 done  |  1 running  |  92 queued
[13:07:33]  51/144 done  |  1 running  |  92 queued
[13:08:38]  51/144 done  |  1 running  |  92 queued
[13:09:45]  51/144 done  |  1 running  |  92 queued
[13:10:51]  51/144 done  |  1 running  |  92 queued
[13:11:57]  51/144 done  |  1 running  |  92 queued
[13:13:03]  51/144 done  |  1 running  |  92 queued
[13:14:08]  51/144 done  |  1 running  |  92 queued
[13:15:15]  51/144 done  |  1 running  |  92 queued
[13:16:21]  52/144 done  |  1 running  |  91 queued
[13:17:26]  52/144 done  |  1 running  |  91 queued
[13:18:31]  

[13:53:59]  54/144 done  |  1 running  |  89 queued
[13:55:08]  54/144 done  |  1 running  |  89 queued
[13:56:15]  54/144 done  |  1 running  |  89 queued
[13:57:20]  54/144 done  |  1 running  |  89 queued
[13:58:26]  54/144 done  |  1 running  |  89 queued
[13:59:33]  54/144 done  |  1 running  |  89 queued
[14:00:43]  54/144 done  |  1 running  |  89 queued
[14:01:48]  54/144 done  |  1 running  |  89 queued
[14:02:57]  55/144 done  |  1 running  |  88 queued
[14:04:02]  55/144 done  |  1 running  |  88 queued
[14:05:09]  55/144 done  |  1 running  |  88 queued
[14:06:17]  55/144 done  |  1 running  |  88 queued
[14:07:21]  55/144 done  |  1 running  |  88 queued
[14:08:27]  55/144 done  |  1 running  |  88 queued
[14:09:32]  55/144 done  |  1 running  |  88 queued
[14:10:38]  55/144 done  |  1 running  |  88 queued
[14:11:44]  55/144 done  |  1 running  |  88 queued
[14:12:49]  55/144 done  |  1 running  |  88 queued
[14:13:55]  55/144 done  |  1 running  |  88 queued
[14:14:59]  

[14:51:38]  57/144 done  |  1 running  |  86 queued
[14:53:05]  57/144 done  |  1 running  |  86 queued
[14:54:11]  57/144 done  |  1 running  |  86 queued
[14:55:17]  57/144 done  |  1 running  |  86 queued
[14:56:24]  57/144 done  |  1 running  |  86 queued
[14:57:30]  57/144 done  |  1 running  |  86 queued
[14:58:40]  57/144 done  |  1 running  |  86 queued
[14:59:48]  57/144 done  |  1 running  |  86 queued
[15:00:55]  57/144 done  |  1 running  |  86 queued
[15:02:02]  57/144 done  |  1 running  |  86 queued
[15:03:07]  57/144 done  |  1 running  |  86 queued
[15:04:13]  57/144 done  |  1 running  |  86 queued
[15:05:18]  57/144 done  |  1 running  |  86 queued
[15:06:23]  57/144 done  |  1 running  |  86 queued
[15:07:29]  58/144 done  |  0 running  |  86 queued
[15:08:35]  58/144 done  |  1 running  |  85 queued
[15:09:43]  58/144 done  |  1 running  |  85 queued
[15:10:48]  58/144 done  |  1 running  |  85 queued
[15:11:53]  58/144 done  |  1 running  |  85 queued
[15:13:01]  

In [ ]:
# ── Cell 7: Consolidate exported files into final dashboard folder ──
import os
import shutil
from glob import glob

def consolidate_event_exports(event_key):
    src_dir = f'/content/drive/MyDrive/CHRS_EarthAI_export_{event_key}'
    dst_dir = os.path.join(DRIVE_DATA_ROOT, event_key)
    os.makedirs(dst_dir, exist_ok=True)

    if not os.path.isdir(src_dir):
        print(f'[{event_key}] source export folder not found: {src_dir}')
        return

    moved = 0
    for src_path in glob(os.path.join(src_dir, '*')):
        fname = os.path.basename(src_path)

        # Keep original exported filename first
        dst_path = os.path.join(dst_dir, fname)
        if not os.path.exists(dst_path):
            shutil.move(src_path, dst_path)
            moved += 1

    print(f'[{event_key}] moved {moved} files → {dst_dir}')

for event_key in EVENTS_TO_RUN:
    consolidate_event_exports(event_key)

In [ ]:
# ── Optional: remove "<event>_" prefix inside final folders ───────
import os

def strip_event_prefix(event_key):
    d = os.path.join(DRIVE_DATA_ROOT, event_key)
    if not os.path.isdir(d):
        return

    prefix = f'{event_key}_'
    for f in os.listdir(d):
        if f.startswith(prefix):
            old_path = os.path.join(d, f)
            new_path = os.path.join(d, f[len(prefix):])
            if not os.path.exists(new_path):
                os.rename(old_path, new_path)

    print(f'[{event_key}] final files: {sorted(os.listdir(d))}')

for event_key in EVENTS_TO_RUN:
    strip_event_prefix(event_key)

## Drive Folder Structure (per event)

```
My Drive/
├── GEE_Flood_harvey/
│   ├── harvey_SAR_before.tif
│   ├── harvey_SAR_after.tif
│   ├── harvey_NDWI_before.tif
│   ├── harvey_NDWI_after.tif
│   ├── harvey_RGB_before.tif
│   ├── harvey_RGB_after.tif
│   ├── harvey_DEM_slope.tif          ← 2-band: elevation, slope
│   ├── harvey_JRC_permanent_water.tif
│   ├── harvey_RF_training_samples.csv
│   └── harvey_GPM_rainfall_daily.csv
├── GEE_Flood_pakistan/   (same structure)
... (13 folders total)
```

## Preparing Files for Streamlit

Strip the `{event}_` prefix and place files in:
```
streamlit_app/data/
├── harvey/
│   ├── SAR_before.tif
│   ├── SAR_after.tif
│   └── ...
├── pakistan/
... (13 subfolders)
```

## Troubleshooting

**No SAR images found** — The `load_sar()` function automatically retries without the orbit direction filter.

**Too few RF samples (< 200)** — Check that the Copernicus EMS asset is uploaded correctly to `projects/chrs-earthai/assets/ems/{EMSR_CODE}_flood`. Verify the flood polygons cover the AOI.

**No EMS ground truth** — If `build_rf_samples` returns None, the EMS asset was not found. Upload the EMS shapefile to the correct GEE asset path (see Cell 3b).

**GeoTIFF tiles** — If exports split into `{name}-0000…0000.tif` files, merge with `rasterio.merge` or increase `SAR_SCALE` / `OPT_SCALE`.

**SKIP_EXISTING = True** — Events whose Drive folder already has ≥ 9 files are skipped automatically. Set `False` to force re-export.